# Do US-listed ETFs forecast markets that are closed?

**The setup.** EWJ is a fund holding ~200 Japanese shares. It trades in New York.

Tokyo closes at 15:30 Japan time — **02:30 in New York**. So when EWJ trades from
09:30 to 16:00 New York time, every share it holds last traded 13.5 hours ago.
Nothing in the basket is moving.

**So any move EWJ makes during the US day is not the basket changing value.**
It's traders revising what they think Japanese equities are worth, on news that
arrived after Tokyo shut. That's a forecast.

This notebook tests whether the forecast comes true.

---

One Tuesday, in New York time:

```
Mon 20:00   Tokyo opens (Tuesday morning there)
Tue 02:30   Tokyo closes        <- basket frozen from here
Tue 09:30   New York opens, EWJ starts trading
Tue 16:00   New York closes     <- our signal is this window
Tue 20:00   Tokyo opens again   <- what we're predicting
```

In [ ]:
# If anything is missing:  pip install yfinance pandas numpy matplotlib
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.width', 120)
print('ready')

## 1. Getting the data

yfinance rate-limits hard, so we save each download to disk and never fetch twice.
If you re-run this notebook it reads the cached CSVs.

**We use raw `Open` and `Close`, never `Adj Close`.** Adjusted close folds dividends
into the price. Mixing an adjusted close with a raw open gives you a fake jump on
every ex-dividend day.

In [ ]:
CACHE = 'data_cache'
START = '2015-01-01'

def get(ticker):
    os.makedirs(CACHE, exist_ok=True)
    path = os.path.join(CACHE, ticker.replace('^', '').replace('=', '') + '.csv')

    if os.path.exists(path):
        df = pd.read_csv(path, index_col=0, parse_dates=True)
        print(f'{ticker}: {len(df)} rows from cache')
    else:
        import yfinance as yf
        df = yf.download(ticker, start=START, progress=False, auto_adjust=False)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        df.to_csv(path)
        print(f'{ticker}: downloaded {len(df)} rows')

    out = df[['Open', 'Close']].dropna()
    out.index = pd.to_datetime(out.index).tz_localize(None).normalize()
    return out

In [ ]:
ewj  = get('EWJ')      # US-listed Japan ETF, dated by New York calendar
n225 = get('^N225')    # Nikkei 225, dated by Tokyo calendar

ewj.tail(3)

In [ ]:
n225.tail(3)

## 2. The signal

EWJ's return from the New York open to the New York close. Tokyo is shut for that
whole window, so this is pure revision of opinion.

We use **log returns** — `log(close / open)` — because they add up cleanly over time
and make the currency correction exact if we add it later.

In [ ]:
ewj_signal = np.log(ewj['Close'] / ewj['Open']).rename('signal')

print(f'{len(ewj_signal)} days')
print(f'typical size: {ewj_signal.std() * 100:.2f}% per day')
ewj_signal.tail(3)

## 3. What we're predicting

Two different things, and the difference matters.

**The gap** — where Tokyo *opens* versus where it last *closed*. If EWJ has already
done the price discovery, the information should land here, in the opening auction.

**The intraday move** — what happens between Tokyo's open and its close. If there's
still a relationship here, Tokyo under-reacted at the open and spent the day catching up.

In [ ]:
n225_gap      = np.log(n225['Open'] / n225['Close'].shift(1)).rename('gap')
n225_intraday = np.log(n225['Close'] / n225['Open']).rename('intraday')

tokyo = pd.concat([n225_gap, n225_intraday], axis=1).dropna()
tokyo.tail(3)

## 4. Alignment — this is where the project lives or dies

On any calendar date `d`:

- Tokyo's session for `d` ran from about 20:00 on `d-1` to 02:30 on `d`, New York time
- New York's session for `d` runs 09:30 to 16:00 on `d`

**So New York's day-`d` session happens AFTER Tokyo's day-`d` session.** The Tokyo
session we want to predict is the *next* one.

If you accidentally match New York on `d` to Tokyo on `d`, you are looking backwards
in time. The result will look wrong in a way that's hard to spot.

`merge_asof` with `direction='forward'` and `allow_exact_matches=False` means:
*match each New York date to the first Tokyo date strictly after it.* That handles
weekends, Japanese holidays and US holidays without us reasoning about any of them —
which matters, because the two calendars are completely independent.

In [ ]:
left = ewj_signal.dropna().reset_index()
left.columns = ['us_date', 'signal']

right = tokyo.reset_index()
right.columns = ['tokyo_date', 'gap', 'intraday']

df = pd.merge_asof(
    left.sort_values('us_date'),
    right.sort_values('tokyo_date'),
    left_on='us_date', right_on='tokyo_date',
    direction='forward',          # the next Tokyo session...
    allow_exact_matches=False,    # ...strictly after, never the same day
).dropna()

# Drop pairs separated by long holiday runs
df['gap_days'] = (df['tokyo_date'] - df['us_date']).dt.days
df = df[df['gap_days'] <= 4]

print(f'{len(df)} matched observations, {df.us_date.min().date()} to {df.us_date.max().date()}')

### Check this by hand before trusting anything below

Pick two or three rows and verify against a calendar. Every `tokyo_date` must be
*after* its `us_date`. Where `gap_days` is 3 you're usually spanning a weekend.

In [ ]:
df[['us_date', 'tokyo_date', 'gap_days', 'signal', 'gap', 'intraday']].head(10)

In [ ]:
# Sanity check that should never fail
assert (df['tokyo_date'] > df['us_date']).all(), 'LOOK-AHEAD BUG'
print('all Tokyo dates are strictly after their US date')
print()
print(df['gap_days'].value_counts().sort_index())

## 5. The result

Fit a straight line through the scatter. The **slope** is the answer:
*for every 1% EWJ moves in New York, how much does Tokyo move next session?*

A slope near 1 means EWJ forecast accurately. Near 0 means no relationship.

The **t-statistic** tells you whether the slope is really different from zero.
Above about 2 is conventionally 'yes'. Ours should be far above that if this works.

**R²** is the fraction of Tokyo's movement explained by EWJ. Don't expect it to be
high — most of what happens overnight is genuinely new information.

In [ ]:
def fit(x, y, label):
    x, y = np.asarray(x), np.asarray(y)
    beta, alpha = np.polyfit(x, y, 1)
    resid = y - (alpha + beta * x)
    se = np.sqrt((resid @ resid) / (len(x) - 2) / ((x - x.mean()) @ (x - x.mean())))
    r2 = np.corrcoef(x, y)[0, 1] ** 2
    print(f'{label:20s} slope {beta:6.3f}   se {se:.3f}   t {beta/se:6.1f}   R2 {r2:.3f}')
    return beta

b_gap      = fit(df['signal'], df['gap'],      'Tokyo opening gap')
b_intraday = fit(df['signal'], df['intraday'], 'Tokyo intraday')

## 6. The picture

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharex=True, sharey=True)

for ax, col, title in zip(axes, ['gap', 'intraday'],
                          ['Tokyo opening gap', 'Tokyo session after the open']):
    ax.scatter(df['signal'] * 100, df[col] * 100, s=6, alpha=0.3,
               edgecolors='none', color='#26356B')
    b, a = np.polyfit(df['signal'], df[col], 1)
    xs = np.linspace(df['signal'].min(), df['signal'].max(), 50)
    ax.plot(xs * 100, (a + b * xs) * 100, color='#A8620C', lw=2, label=f'slope {b:.2f}')
    ax.axhline(0, color='#bbb', lw=0.6)
    ax.axvline(0, color='#bbb', lw=0.6)
    ax.set_xlabel('EWJ return during US session (%)')
    ax.set_title(title)
    ax.legend(frameon=False)

axes[0].set_ylabel('Nikkei return (%)')
fig.suptitle('Does a US-listed Japan ETF forecast the next Tokyo session?', y=1.02)
fig.tight_layout()
plt.show()

## 7. Reading the result

**Slope near 1 on the gap, near 0 on the intraday** — EWJ forecast well and Tokyo
priced it in at the open. The efficient outcome, and the one I'd bet on.

**Slope well below 1 on the gap, positive on the intraday** — Tokyo under-reacts at
the open and drifts during the day. More interesting, and worth more in a write-up.

**Slope near 0 on both** — almost certainly a bug rather than a finding. Go back to
section 4 and check the dates by hand.

---

### What we deliberately left out

**Currency.** EWJ is priced in dollars, so part of its move is the yen moving rather
than Japanese equities repricing. This adds noise; it can't manufacture a fake result.

**Index mismatch.** EWJ tracks MSCI Japan, not the Nikkei 225. Different constituents,
different weighting. TOPIX would be closer.

Both are honest limitations to state in the write-up, and both are v2.